__Log Likelihood Function: Pixelization__

This script provides a step-by-step guide of the **PyAutoGalaxy** `log_likelihood_function` which is used to fit
`Interferometer` data with an inversion (specifically a `RectangularUniform` mesh and `Constant` regularization scheme`).

This script has the following aims:

 - To provide a resource that authors can include in papers using **PyAutoGalaxy**, so that readers can understand the
 likelihood function (including references to the previous literature from which it is defined) without having to
 write large quantities of text and equations.

 - To make inversions in **PyAutoGalaxy** less of a "black-box" to users.

__Prerequisites__

The likelihood function of pixelizations is the most complicated likelihood function.

The likelihood function of a pixelization builds on that used for standard parametric light profiles and
linear light profiles, therefore you must read the following notebooks before this script:

- `interferometer/likelihood_function.ipynb`.
- `interferometer/features/linear_light_profiles/likelihood_function.ipynb`.

This script repeats all text and code examples in the above likelihood function examples. It therefore can be used to
learn about the linear light profile likelihood function without reading other likelihood scripts.

__Contents__

- **Mesh Shape:** Fixing the rectangular mesh shape before modeling.
- **Mask:** Defining the real-space mask for the interferometer grid.
- **Dataset:** Loading the interferometer dataset from FITS files.
- **Dataset Auto-Simulation:** Automatically simulating data if it does not exist.
- **Over Sampling:** Why over sampling is not needed for interferometer data.
- **Masked Image Grid:** Setting up the 2D coordinate grid for pixelization evaluation.
- **Galaxy:** Combining the pixelization into a Galaxy object.
- **Rectangular Mesh:** Creating the rectangular mesh overlaid on the masked image grid.
- **Interpolation:** Creating an interpolator describing how image pixels map to mesh pixels.
- **Mapper:** Creating the Mapper object that describes image-to-reconstruction pixel mappings.
- **Over Sampling:** How over sampling interacts with the pixelization mapping scheme.
- **Alternative Meshes:** Brief overview of other mesh types available.
- **Mapping Matrix:** Constructing the 2D mapping matrix from image pixels to reconstruction pixels.
- **Transformed Mapping Matrix ($f$):** Fourier transforming the mapping matrix to the uv-plane.
- **Data Vector (D):** Computing the data vector for the linear inversion.
- **Curvature Matrix (F):** Computing the curvature matrix for the linear inversion.
- **Regularization Matrix (H):** Constructing the regularization matrix to enforce smoothness.
- **F + Lamdba H:** Combining the curvature and regularization matrices.
- **Galaxy Reconstruction (s):** Solving the linear system to reconstruct the galaxy.
- **Visibilities Reconstruction:** Mapping the reconstruction back to the uv-plane.
- **Likelihood Function:** Defining the full Bayesian log likelihood for pixelized modeling.
- **Chi Squared:** Computing the chi-squared statistic from residuals and noise.
- **Regularization Term:** The regularization penalty term in the log likelihood.
- **Complexity Terms:** The log determinant terms that penalize overly complex reconstructions.
- **Noise Normalization Term:** The noise normalization constant in the log likelihood.
- **Calculate The Log Likelihood:** Combining all terms to compute the final log evidence.
- **Fit:** Performing the likelihood evaluation using the FitInterferometer object.
- **Galaxy Modeling:** Overview of how the likelihood function is sampled by a non-linear search.
- **Log Likelihood Function: Pixelization With Light Profile:** How the likelihood changes when parametric light profiles are included.
- **Log Likelihood Function: Fast Chi Squared:** Computing chi-squared without NUFFT for speed.
- **Wrap Up:** Summary and pointers to additional resources.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import autogalaxy as ag
import autogalaxy.plot as aplt
from autoarray.inversion.plot.mapper_plots import plot_mapper, subplot_image_and_mapper

__Mesh Shape__

As discussed in the `features/pixelization/modeling` example, the mesh shape is fixed before modeling.

In [ ]:
mesh_pixels_yx = 28
mesh_shape = (mesh_pixels_yx, mesh_pixels_yx)

__Mask__

We define the ‘real_space_mask’ which defines the grid the image the galaxy is evaluated using.

In [ ]:
real_space_mask = ag.Mask2D.circular(
    shape_native=(80, 80), pixel_scales=0.05, radius=4.0
)

__Dataset__

Load and plot the galaxy `Interferometer` dataset `simple` from .fits files, which we will fit 
with the model.

This includes the method used to Fourier transform the real-space image of the galaxy to the uv-plane and compare 
directly to the visibilities. We use a non-uniform fast Fourier transform, which is the most efficient method for 
interferometer datasets containing ~1-10 million visibilities. We will discuss how the calculation of the likelihood
function changes for different methods of Fourier transforming in this guide.

In [ ]:
dataset_name = "clumpy"
dataset_path = Path("dataset") / "interferometer" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if ag.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/interferometer/features/pixelization/simulator.py"],
        check=True,
    )


dataset = ag.Interferometer.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    uv_wavelengths_path=dataset_path / "uv_wavelengths.fits",
    real_space_mask=real_space_mask,
    transformer_class=ag.TransformerNUFFT,
)

This guide uses in-built visualization tools for plotting. 

For example, using the `Interferometer` the dataset we perform a likelihood evaluation on is plotted.

The `subplot_dataset` displays the visibilities in the uv-plane, which are the raw data of the interferometer
dataset. These are what will ultimately be directly fitted in the Fourier space.

The `subplot_dirty_images` displays the dirty images of the dataset, which are the reconstructed images of visibilities
using an inverse Fourier transform to convert these to real-space. These dirty images are not the images we fit, but
visualization of the dirty images are often used in radio interferometry to show the data in a way that is more
interpretable to the human eye.

In [ ]:
aplt.subplot_interferometer_dirty_images(dataset=dataset)

__Over Sampling__

Over sampling evaluates a light profile using multiple samples of its intensity per image-pixel.

For interferometer datasets, over sampling is not used in the pixelization (or for light profiles)
therefore it is implicitly set to 1 and can be ignored hereafter.

For CCD imaging datasets, over sampling is normally used and slightly changes the likelihood function.

__Masked Image Grid__

To perform galaxy calculations we define a 2D image-plane grid of (y,x) coordinates.

For light profiles these are given by `dataset.lp`, which is a uniform grid of (y,x) Cartesian coordinates
which have had the 3.0" circular mask applied.

A pixelization uses a separate grid of (y,x) coordinates, called `dataset.grids.pixelization`, which is
identical to the light profile grid but may of had a different over-sampling scale applied (but in this example
does not).

Each (y,x) coordinate coordinates to the centre of each image-pixel in the dataset, meaning that when this grid is
used to construct a pixelization there is a straight forward mapping between the image data and pixelization pixels.

In [ ]:
aplt.plot_grid(grid=dataset.grids.pixelization, title="Grid")

__Galaxy__

We combine the pixelization into a single `Galaxy` object.

The galaxy includes the rectangular mesh and constant regularization scheme, which will ultimately be used
to reconstruct its star forming clumps.

In [ ]:
pixelization = ag.Pixelization(
    mesh=ag.mesh.RectangularUniform(shape=(30, 30)),
    regularization=ag.reg.Constant(coefficient=1.0),
)

galaxy = ag.Galaxy(redshift=0.5, pixelization=pixelization)

__Rectangular Mesh__

The pixelization is used to create the rectangular mesh which is used to reconstruct the galaxy.

The function below does this by overlaying the rectangular mesh over the masked image grid, such that the edges of
the rectangular mesh touch the ask grid's edges.

In [ ]:
from autoarray.inversion.mesh.mesh.rectangular_rtu_adapt_density import (
    overlay_grid_from,
)

mesh_grid = overlay_grid_from(
    shape_native=galaxy.pixelization.mesh.shape, grid=dataset.grids.pixelization
)

__Interpolation__

We now combine grids computed above to create an `Interpolator`, which describes how image grid pixel maps to
every rectangular mesh pixel. 

In [ ]:
interpolator = pixelization.mesh.interpolator_from(
    source_plane_data_grid=dataset.grids.pixelization,
    source_plane_mesh_grid=mesh_grid,
)

For the rectangular mesh, the interpolation scheme is called bilinear interpolation, which means that every image 
pixel maps to the rectangular pixel it lands in and the three neighboring rectangular pixels. 

The weight of each mapping is determined by the bilinear interpolation scheme, which is a function of how close the 
image pixel is to the centre of the rectangular pixel it lands in and the three neighboring rectangular pixels.

We can print the mappings and weights, for example of the first image pixel, to confirm this.

In [ ]:
print(interpolator.mappings[0])
print(interpolator.weights[0])

__Mapper__

The rectangular mesh defines the plane in which the galaxy's surface brightness is reconstructed. In the library
API this plane is called the `source-plane` for historical reasons (the same mesh-based inversion machinery is used
to reconstruct lensed source galaxies in PyAutoLens). For PyAutoGalaxy there is no lensing, but the attribute names
on the `Mapper` keep the lens-modelling naming so the implementation can be shared between the two libraries.

We now use the interpolator to create a `Mapper`, which describes the mapping between every image pixel and every 
rectangular pixel, based on the interpolation scheme above.

In [ ]:
mapper = ag.Mapper(interpolator=interpolator)

Plotting the rectangular mesh shows that the reconstruction plane has been discretized into a grid of rectangular
pixels.

Below, we plot the rectangular mesh without the image-grid pixels (for clarity) and with them as black dots in order
to show how each set of image-pixels fall within a rectangular pixel.

In [ ]:
plot_mapper(mapper=mapper)

plot_mapper(mapper=mapper, mesh_grid=mapper.source_plane_data_grid)

The `Mapper` contains:

 1) `source_plane_data_grid`: the grid of masked (y,x) image-pixel coordinate centres (`masked_dataset.grids.pixelization`).
 2) `source_plane_mesh_grid`: The rectangular mesh of (y,x) pixelization pixel coordinates (`mesh_grid`).

We have therefore discretized the reconstruction plane into a rectangular mesh, and can pair every image-pixel
coordinate with the corresponding rectangular pixel it lands in.

These quantities are both in the reconstruction plane, and do not by themselves describe the mapping between the
image and reconstruction. The mapping is described by the `pix_indexes_for_sub_slim_index`, which maps every
image-pixel index to every pixelization pixel index.

`pix_indexes` refers to the pixelization pixel indexes (e.g. rectangular pixel 0, 1, 2 etc.) and `sub_slim_index`  
refers to the index of an image pixel (e.g. image-pixel 0, 1, 2 etc.). 

For example, printing the first ten entries of `pix_indexes_for_sub_slim_index` shows the first ten rectangular
pixelization pixel indexes these image sub-pixels map too.

In [ ]:
pix_indexes_for_sub_slim_index = mapper.pix_indexes_for_sub_slim_index

print(pix_indexes_for_sub_slim_index[0:9])

This array can be used to visualize how an input list of image-pixel indexes map to the rectangular pixelization.

It also shows that image-pixel indexing begins from the top-left and goes rightwards and downwards, accounting for 
all image-pixels which are not masked.

In [ ]:
subplot_image_and_mapper(mapper=mapper, image=dataset.dirty_image)

The reverse mapping, from a reconstruction pixel to the image-pixels which map into it, is given by
`mapper.mappings_from`. It takes a list of reconstruction pixel index groups and returns one `Mapping` per group,
carrying the outline of the reconstruction cell (`source_contours`) and the outlines of the image-pixel regions which
map into it (`image_contours`), both as polygons in arc-seconds.

Passing the mappings to `subplot_image_and_mapper` as `regions=` draws them in matched colours, the cell on the right
and the image-pixels it owns on the left.

In [ ]:
pix_indexes = [[min(200, mapper.pixels - 1)]]

mappings = mapper.mappings_from(pix_indexes=pix_indexes)

subplot_image_and_mapper(mapper=mapper, image=dataset.dirty_image, regions=mappings)

__Interpolation__

The left hand plot shows a region wider than the handful of image pixels which land strictly inside reconstruction
pixel 200. Pixels further out than the pixel are also mapped to this reconstruction pixel.

This is because of the interpolation mapping scheme whereby each image pixel is paired with four reconstruction
pixels.

We can confirm that every image pixel maps to four reconstruction pixels by printing 
the `pix_sizes_for_sub_slim_index`, which gives the number of mapped reconstruction pixels for every image pixel.

We can also confirm that the interpolation introduces weights to each mapping by printing the 
`pix_weights_for_sub_slim_index`, which gives the weight of each mapping for every image pixel.

In [ ]:
print(mapper.pix_sizes_for_sub_slim_index[0:9])
print(mapper.pix_weights_for_sub_slim_index[0:9])

__Over Sampling__

Lets quickly think about what happens when we use over sampling in the pixelization (e.g. `sub_size>1`). For
the `sub_size=1` case above, each image pixel maps to 4 reconstruction pixels (due to bilinear interpolation)
with a weight determined from the bilinear interpolation scheme.

However, the default over sampling for a pixelization is `sub_size=4`, meaning each image pixel is divided
into a 4x4 grid of sub-pixels (16 sub-pixels in total). Each of these sub-pixels maps to 4 reconstruction pixels
(due to bilinear interpolation), where the weight of each mapping is determined by the bilinear interpolation
scheme divided by 16 (because there are 16 sub-pixels).

This example therefore used a `sub_size=1` to keep the explanation of the likelihood, visualization of the
arrays above and understanding of the mapping scheme as simple as possible. You can manually increase the
`sub_size` above and re-run the notebook to see how this changes the mapping scheme.

__Alternative Meshes__

We can briefly consider how this step differs for other mesh types. Above, we simply overlaid a uniform rectangular
grid to define the reconstruction pixel centres and then mapped image pixels to these reconstruction pixels.

The `RectangularUniform` mesh pretty much works exactly the same, its just that a calculation (which we don't
describe here) works out how to make a grid of rectangular pixels that adapt to the data density and thus
vary in size.

There is also a `RectangularBilinearAdaptImage` mesh which uses the image of the galaxy to adapt
the rectangular pixel sizes. This often puts even smaller pixels in the brightest regions of the galaxy,
even if it lies offset or away from the caustic.

There is also a `Delaunay` mesh which uses a Delaunay triangulation to define an irregular grid of reconstruction pixels.
This is described fully in the `delaunay` example including a likelihood function guide.

__Mapping Matrix__

The `mapping_matrix` represents the image-pixel to reconstruction-pixel mappings above in a 2D matrix.

It has dimensions `(total_image_pixels, total_rectangular_pixels)`.

In [ ]:
mapping_matrix = ag.util.mapper.mapping_matrix_from(
    pix_indexes_for_sub_slim_index=pix_indexes_for_sub_slim_index,
    pix_size_for_sub_slim_index=mapper.pix_sizes_for_sub_slim_index,
    pix_weights_for_sub_slim_index=mapper.pix_weights_for_sub_slim_index,
    pixels=mapper.pixels,
    total_mask_pixels=mapper.source_plane_data_grid.mask.pixels_in_mask,
    slim_index_for_sub_slim_index=mapper.slim_index_for_sub_slim_index,
    sub_fraction=mapper.over_sampler.sub_fraction,
)

A 2D plot of the `mapping_matrix` shows of all image-pixelization pixel mappings.

No row of pixels has more than one non-zero entry. It is not possible for two image pixels to map to the same 
pixelization pixel (meaning that there are no correlated pixels in the mapping matrix).

In [ ]:
plt.imshow(mapping_matrix, aspect=(mapping_matrix.shape[1] / mapping_matrix.shape[0]))
plt.show()
plt.close()

Each column of the `mapping_matrix` can therefore be used to show all image-pixels it maps too. 

For example, above, we plotted all image-pixels of pixelization pixel 200 (as well as 202 and 204). We can extract all
image-pixel indexes of pixelization pixels 200 using the `mapping_matrix` and use them to plot the image of this
pixelization pixel (which corresponds to only values of zeros or ones).

In [ ]:
indexes_pix_200 = np.nonzero(mapping_matrix[:, 200])

print(indexes_pix_200[0])

array_2d = ag.Array2D(values=mapping_matrix[:, 200], mask=dataset.mask)

aplt.plot_array(array=array_2d, title="Mapping Matrix Column")

__Transformed Mapping Matrix ($f$)__

Each pixelization pixel can therefore be thought of as an image (where all entries of this image are zeros and ones). 

However, for interferometer datasets we want to fit the visibilities in the uv-plane, not the image-plane. Therefore,
each image in the `mapping_matrix` must be transformed to the uv-plane via a Fourier transform, such that each
column in the `transformed_mapping_matrix` represents the visibilities in the uv-plane of each pixelization pixel.

This operation changes the dimensions of the mapping matrix, meaning the `transformed_mapping_matrix` has
dimensions `(total_image_pixels, total_visibilities)`. 

If the number of visibilities is large (e.g. 10^6) this matrix becomes extremely large and computationally expensive to 
store memory, meaning the sparse operator matrices must be used instead.

The `transformed_mapping_matrix` is also complex, storing all entries of the visibilities after the NUFFT as real
and complex values.

In [ ]:
transformed_mapping_matrix = dataset.transformer.transform_mapping_matrix(
    mapping_matrix=mapping_matrix
)

A 2D plot of the `transformed_mapping_matrix` shows all visibility-reconstruction pixel mappings.

Note how, unlike for the `mapping_matrix`, every row of image-pixels fully consists of non-zero entries. This
means the matrix is fully dense, making it even more difficult to store in memory for large datasets.

Below, we plot the real and imaginary components of the `transformed_mapping_matrix` separately.

In [ ]:
plt.imshow(
    transformed_mapping_matrix.real,
    aspect=(transformed_mapping_matrix.shape[1] / transformed_mapping_matrix.shape[0]),
)
plt.colorbar()
plt.show()
plt.close()

plt.imshow(
    transformed_mapping_matrix.imag,
    aspect=(transformed_mapping_matrix.shape[1] / transformed_mapping_matrix.shape[0]),
)
plt.colorbar()
plt.show()
plt.close()

Each column of the `transformed_mapping_matrix` shows all visibilities it maps to after the NUFFT.

In [ ]:
indexes_pix_200 = np.nonzero(transformed_mapping_matrix[:, 200])

print(indexes_pix_200[0])

visibilities = ag.Visibilities(visibilities=transformed_mapping_matrix[:, 200])

aplt.plot_grid(grid=visibilities.in_grid, title="Visibilities")

In Warren & Dye 2003 (https://arxiv.org/abs/astro-ph/0302587) the `transformed_mapping_matrix` is denoted $f_{ij}$
where $i$ maps over all $I$ reconstruction pixels and $j$ maps over all $J$ visibilities. 

For example: 

 - $f_{0, 2} = 0.3$ indicates that visibility number $2$ maps to pixelization pixel $0$ with a weight of $0.3$ after the NUFFT.

The indexing of the `mapping_matrix` is reversed compared to the notation of WD03 (e.g. visibilities
are the first entry of `mapping_matrix` whereas for $f$ they are the second index).

In [ ]:
print(f"Mapping between visibility 0 and rectangular pixel 2 = {mapping_matrix[0, 2]}")

__Data Vector (D)__

To solve for the rectangular pixel fluxes we now pose the problem as a linear inversion.

This requires us to convert the `transformed_mapping_matrix` and our `data` and `noise map` into matrices of certain dimensions. 

The `data_vector`, $D$, is the first matrix and it has dimensions `(total_rectangular_pixels,)`.

In WD03 (https://arxiv.org/abs/astro-ph/0302587) and N15 (https://arxiv.org/abs/1412.7436) the data vector 
is give by: 

 $\vec{D}_{i} = \sum_{\rm  j=1}^{J}f_{ij}(d_{j})/\sigma_{j}^2 \, \, .$

Where:

 - $d_{\rm j}$ are the image-pixel data flux values.
 - $\sigma{\rm _j}^2$ are the statistical uncertainties of each image-pixel value.

$i$ maps over all $I$ reconstruction pixels and $j$ maps over all $J$ image pixels. 

In [ ]:
data_vector = (
    ag.util.inversion_interferometer.data_vector_via_transformed_mapping_matrix_from(
        transformed_mapping_matrix=transformed_mapping_matrix,
        visibilities=dataset.data,
        noise_map=dataset.noise_map,
    )
)

$D$ describes which rectangular pixels trace to which visibilities, with associated weights, after the NUFFT. This 
ensures the reconstruction fully accounts for the NUFFT when fitting the data.

We can plot $D$ as a column vector:

In [ ]:
plt.imshow(
    data_vector.reshape(data_vector.shape[0], 1), aspect=10.0 / data_vector.shape[0]
)
plt.colorbar()
plt.show()
plt.close()

The dimensions of $D$ are the number of reconstruction pixels.

In [ ]:
print("Data Vector:")
print(data_vector)
print(data_vector.shape)

__Curvature Matrix (F)__

The `curvature_matrix` $F$ is the second matrix and it has 
dimensions `(total_rectangular_pixels, total_rectangular_pixels)`.

In WD03 / N15 (https://arxiv.org/abs/astro-ph/0302587) the curvature matrix is a 2D matrix given by:

 ${F}_{ik} = \sum_{\rm  j=1}^{J}f_{ij}f_{kj}/\sigma_{j}^2 \, \, .$

NOTE: this notation implicitly assumes a summation over $K$, where $k$ runs over all pixelization pixel indexes $K$.

Note how summation over $J$ runs over $f$ twice, such that every entry of $F$ is the sum of the multiplication
between all values in every two columns of $f$.

For example, $F_{0,1}$ is the sum of all visibility values in $f$ of reconstruction pixel 0 multiplied by
all visibility values of reconstruction pixel 1.

Visibilities are both real and complex values, and the `curvature_matrix` is computed separately for the real and
imaginary components of the visibilities and then summed together.

In [ ]:
real_curvature_matrix = ag.util.inversion.curvature_matrix_via_mapping_matrix_from(
    mapping_matrix=transformed_mapping_matrix.real,
    noise_map=dataset.noise_map.real,
)

imag_curvature_matrix = ag.util.inversion.curvature_matrix_via_mapping_matrix_from(
    mapping_matrix=transformed_mapping_matrix.imag,
    noise_map=dataset.noise_map.imag,
)

curvature_matrix = np.add(real_curvature_matrix, imag_curvature_matrix)

plt.imshow(curvature_matrix)
plt.colorbar()
plt.show()
plt.close()

For $F_{ik}$ to be non-zero, this requires that the images of rectangular pixels $i$ and $k$ share at least one
image-pixel, which for visibilities after the NUFFT is always true for all $i$ and $k$.

For example, we can see a non-zero entry for $F_{100,101}$ and plotting their images
show overlap.

In [ ]:
recon_pixel_0 = 0
recon_pixel_1 = 1

print(curvature_matrix[recon_pixel_0, recon_pixel_1])

visibilities = ag.Visibilities(
    visibilities=transformed_mapping_matrix[:, recon_pixel_0],
)

aplt.plot_grid(grid=visibilities.in_grid, title="Visibilities")

visibilities = ag.Visibilities(
    visibilities=transformed_mapping_matrix[:, recon_pixel_1],
)

aplt.plot_grid(grid=visibilities.in_grid, title="Visibilities")

The following chi-squared is minimized when we perform the inversion and reconstruct the galaxy:

$\chi^2 = \sum_{\rm  j=1}^{J} \bigg[ \frac{(\sum_{\rm  i=1}^{I} s_{i} f_{ij}) - d_{j}}{\sigma_{j}} \bigg]$

Where $s$ is the reconstructed pixel fluxes in all $I$ rectangular pixels.

The solution for $s$ is therefore given by (equation 5 WD03):

 $s = F^{-1} D$

We can compute this using NumPy linear algebra:

In [ ]:

# Because we are no using regularizartion (see below) it is common for the curvature matrix to be singular and lead
# to a LinAlgException. The loop below mitigates this -- you can ignore it as it is not important for understanding
# the PyAutoGalaxy likelihood function.

for i in range(curvature_matrix.shape[0]):
    curvature_matrix[i, i] += 1e-8

reconstruction = np.linalg.solve(curvature_matrix, data_vector)

We can plot this reconstruction -- it looks like a mess.

The pixelization pixels have noisy and unsmooth values, and it is hard to make out if a galaxy is even being 
reconstructed. 

In fact, the linear inversion is (over-)fitting noise in the image data, meaning this system of equations is 
ill-posed. We need to apply some form of smoothing on the reconstruction to avoid over fitting noise.

In [ ]:
plot_mapper(mapper=mapper, solution_vector=reconstruction)

__Regularization Matrix (H)__

Regularization adds a linear regularization term $G_{\rm L}$ to the $\chi^2$ we solve for giving us a new merit 
function $G$ (equation 11 WD03):

 $G = \chi^2 + \lambda \, G_{\rm L}$

where $\lambda$ is the `regularization_coefficient` which describes the magnitude of smoothness that is applied. A 
higher $\lambda$ will regularize the reconstruction more, leading to a smoother galaxy reconstruction.

Different forms for $G_{\rm L}$ can be defined which regularize the reconstruction in different ways. The 
`Constant` regularization scheme used in this example applies gradient regularization (equation 14 WD03):

 $G_{\rm L} = \sum_{\rm  i}^{I} \sum_{\rm  n=1}^{N}  [s_{i} - s_{i, v}]$

This regularization scheme is easier to express in words -- the summation goes to each rectangular pixelization pixel,
determines all rectangular pixels with which it shares a direct vertex (e.g. its neighbors) and penalizes solutions 
where the difference in reconstructed flux of these two neighboring pixels is large.

The summation does this for all rectangular pixels, thus it favours solutions where neighboring rectangular 
pixels reconstruct similar values to one another (e.g. it favours a smooth galaxy reconstruction).

We now define the `regularization matrix`, $H$, which allows us to include this smoothing when we solve for $s$. $H$
has dimensions `(total_rectangular_pixels, total_rectangular_pixels)`.

This relates to $G_{\rm L}$ as (equation 13 WD03):

 $H_{ik} = \frac{1}{2} \frac{\partial G_{\rm L}}{\partial s_{i} \partial s_{k}}$

$H$ has the `regularization_coefficient` $\lambda$ folded into it such $\lambda$'s control on the degree of smoothing
is accounted for.

In [ ]:
regularization_matrix = ag.util.regularization.constant_regularization_matrix_from(
    coefficient=galaxy.pixelization.regularization.coefficient,
    neighbors=mapper.neighbors,
    neighbors_sizes=mapper.neighbors.sizes,
)

We can plot the regularization matrix and note that:

 - non-zero entries indicate that two rectangular pixelization pixels are neighbors and therefore are regularized 
 with one another.

 - Zeros indicate the two rectangular pixels do not neighbor one another.

The majority of entries are zero, because the majority of rectangular pixels are not neighbors with one another.

In [ ]:
plt.imshow(regularization_matrix)
plt.colorbar()
plt.show()
plt.close()

__F + Lamdba H__

$H$ enters the linear algebra system we solve for as follows (WD03 equation (12)):

 $s = [F + H]^{-1} D$

In [ ]:
curvature_reg_matrix = np.add(curvature_matrix, regularization_matrix)

__Galaxy Reconstruction (s)__

We can now solve the linear system above using NumPy linear algebra. 

Note that the for loop used above to prevent a LinAlgException is no longer required.

In [ ]:
reconstruction = np.linalg.solve(curvature_reg_matrix, data_vector)

By plotting this galaxy reconstruction we can see that regularization has lead us to reconstruct a smoother galaxy,
which actually looks like the star forming clumps in the galaxy imaging data! 

This also implies we are not over-fitting the noise.

In [ ]:
plot_mapper(mapper=mapper, solution_vector=reconstruction)

__Visibilities Reconstruction__

Using the reconstructed pixel fluxes we can map the reconstruction back to the image plane (via 
the `blurred mapping_matrix`) and produce a reconstruction of the image data.

In [ ]:
mapped_reconstructed_visibilities = (
    ag.util.inversion_interferometer.mapped_reconstructed_visibilities_from(
        transformed_mapping_matrix=transformed_mapping_matrix,
        reconstruction=reconstruction,
    )
)

mapped_reconstructed_visibilities = ag.Visibilities(
    visibilities=mapped_reconstructed_visibilities
)

aplt.plot_grid(
    grid=mapped_reconstructed_visibilities.in_grid,
    title="Mapped Reconstructed Visibilities",
)


__Likelihood Function__

We now quantify the goodness-of-fit of our pixelization galaxy reconstruction. 

We compute the `log_likelihood` of the fit, which is the value returned by the `log_likelihood_function`.

The likelihood function for galaxy modeling consists of five terms:

 $-2 \mathrm{ln} \, \epsilon = \chi^2 + s^{T} H s + \mathrm{ln} \, \left[ \mathrm{det} (F + H) \right] - { \mathrm{ln}} \, \left[ \mathrm{det} (H) \right] + \sum_{\rm  j=1}^{J} { \mathrm{ln}} \left [2 \pi (\sigma_j)^2 \right]  \, .$

This expression was first derived by Suyu 2006 (https://arxiv.org/abs/astro-ph/0601493) and is given by equation (19).
It was derived into **PyAutoGalaxy** notation in Dye 2008 (https://arxiv.org/abs/0804.4002) equation (5).

We now explain what each of these terms mean.

__Chi Squared__

The first term is a $\chi^2$ statistic, which is defined above in our merit function as and is computed as follows:

 - `model_data` = `mapped_reconstructed_visibilities`
 - `residual_map` = (`data` - `model_data`)
 - `normalized_residual_map` = (`data` - `model_data`) / `noise_map`
 - `chi_squared_map` = (`normalized_residuals`) ** 2.0 = ((`data` - `model_data`)**2.0)/(`variances`)
 - `chi_squared` = sum(`chi_squared_map`)

The chi-squared therefore quantifies if our fit to the data is accurate or not. 

High values of chi-squared indicate that there are many image pixels our model did not produce a good fit to the image 
for, corresponding to a fit with a lower likelihood.

In [ ]:
model_visibilities = mapped_reconstructed_visibilities

residual_map = dataset.data - model_visibilities


normalized_residual_map_real = (residual_map.real / dataset.noise_map.real).astype(
    "complex128"
)
normalized_residual_map_imag = (residual_map.imag / dataset.noise_map.imag).astype(
    "complex128"
)
normalized_residual_map = (
    normalized_residual_map_real + 1j * normalized_residual_map_imag
)


chi_squared_map_real = (residual_map.real / dataset.noise_map.real) ** 2
chi_squared_map_imag = (residual_map.imag / dataset.noise_map.imag) ** 2
chi_squared_map = chi_squared_map_real + 1j * chi_squared_map_imag


chi_squared_real = np.sum(chi_squared_map.real)
chi_squared_imag = np.sum(chi_squared_map.imag)
chi_squared = chi_squared_real + chi_squared_imag

print(chi_squared)

The `chi_squared_map` indicates which regions of the image we did and did not fit accurately.

In [ ]:
chi_squared_map = ag.Visibilities(visibilities=chi_squared_map)

aplt.plot_grid(grid=chi_squared_map.in_grid, title="Chi-Squared Map")

__Regularization Term__

The second term, $s^{T} H s$, corresponds to the $\lambda $G_{\rm L}$ regularization term we added to our merit 
function above.

This is the term which sums up the difference in flux of all reconstructed rectangular pixels, and reduces the 
likelihood of solutions where there are large differences in flux (e.g. the galaxy is less smooth and more likely to be 
overfitting noise).

We compute it below via matrix multiplication, noting that the `regularization_coefficient`, $\lambda$, is built into 
the `regularization_matrix` already.

In [ ]:
regularization_term = np.matmul(
    reconstruction.T, np.matmul(regularization_matrix, reconstruction)
)

print(regularization_term)

__Complexity Terms__

Up to this point, it is unclear why we chose a value of `regularization_coefficient=1.0`. 

We cannot rely on the `chi_squared` and `regularization_term` above to optimally choose its value, because increasing 
the `regularization_coefficient` smooths the solution more and therefore:

 - Decreases `chi_squared` by fitting the data worse, producing a lower `log_likelihood`.

 - Increases the `regularization_term` by penalizing the differences between reconstruction pixel fluxes more, again reducing
 the inferred `log_likelihood`.

If we set the regularization coefficient based purely on these two terms, we would set a value of 0.0 and be back where
we started over-fitting noise!

The terms $\left[ \mathrm{det} (F + H) \right]$ and $ - { \mathrm{ln}} \, \left[ \mathrm{det} (H) \right]$ address 
this problem. 

They quantify how complex the reconstruction is, and penalize solutions where *it is more complex*. Reducing 
the `regularization_coefficient` makes the galaxy reconstruction more complex (because a galaxy that is 
smoothed less uses more flexibility to fit the data better).

These two terms therefore counteract the `chi_squared` and `regularization_term`, so as to attribute a higher
`log_likelihood` to solutions which fit the data with a more smoothed and less complex reconstruction (e.g. one with a higher 
`regularization_coefficient`).

In **HowToGalaxy** -> `chapter 3` -> `tutorial_4_bayesian_regularization` we expand on this further and give a more
detailed description of how these different terms impact the `log_likelihood_function`. 

In [ ]:
log_curvature_reg_matrix_term = np.linalg.slogdet(curvature_reg_matrix)[1]
log_regularization_matrix_term = np.linalg.slogdet(regularization_matrix)[1]

print(log_curvature_reg_matrix_term)
print(log_regularization_matrix_term)

__Noise Normalization Term__

Our likelihood function assumes the imaging data consists of independent Gaussian noise in every image pixel.

The final term ins the likelihood function is therefore a `noise_normalization` term, which consists of the sum
of the log of every noise-map value squared. 

Given the `noise_map` is fixed, this term does not change during the galaxy modeling process and has no impact on the 
model we infer.

In [ ]:
noise_normalization_real = np.sum(np.log(2 * np.pi * dataset.noise_map.real**2.0))
noise_normalization_imag = np.sum(np.log(2 * np.pi * dataset.noise_map.imag**2.0))
noise_normalization = noise_normalization_real + noise_normalization_imag

__Calculate The Log Likelihood__

We can now, finally, compute the `log_likelihood` of the galaxy model, by combining the five terms computed above using
the likelihood function defined above.

In [ ]:
log_evidence = float(
    -0.5
    * (
        chi_squared
        + regularization_term
        + log_curvature_reg_matrix_term
        - log_regularization_matrix_term
        + noise_normalization
    )
)

print(log_evidence)

__Fit__

This process to perform a likelihood function evaluation performed via the `FitInterferometer` object.

In [ ]:
galaxies = ag.Galaxies(galaxies=[galaxy])

fit = ag.FitInterferometer(
    dataset=dataset,
    galaxies=galaxies,
    settings=ag.Settings(use_border_relocator=True),
)
fit_log_evidence = fit.log_evidence
print(fit_log_evidence)

aplt.subplot_fit_interferometer(fit=fit)

__Galaxy Modeling__

To fit a galaxy model to data, the likelihood function illustrated in this tutorial is sampled using a
non-linear search algorithm.

The default sampler is the nested sampling algorithm `nautilus` (https://github.com/johannesulf/nautilus)
multiple MCMC and optimization algorithms are supported.

__Log Likelihood Function: Pixelization With Light Profile__

The above example does not include a parametric light profile in the galaxy model, which the modeling example
shows may be useful if you want to fit both the smooth and clumpy components of a galaxy's light.

A step-by-step guide to the likelihood function is not provided, but in a nutshell one of the following two 
things happen:

- `standard light profile`: If normal light profiles are included in the galaxy model, their light is computed 
  on the image grid, subtracted from the data, and the pixelization is used to reconstruct the resulting light
  profile subtracted image. You can basically think of the pixelization as fitting the residuals left over after
  the light profile has been subtracted.

- `linear light profile`: If linear light profiles are included in the galaxy model, their light is computed
  on the image grid and linearly solved for simultaneously with the pixelization reconstruction. This means
  they trade off their solved for `intensity` values with the reconstruction pixel fluxes to fit the image data. The
  inversion is thus performed simultaneously for both the linear light profile intensities and reconstruction pixel fluxes!

__Log Likelihood Function: Fast Chi Squared__

This script describes how the `chi_squared` of an interferometer pixelization can be computed without using a
`transformed_mapping_matrix` or an NUFFT algorithm at all.

This means the likelihood function can be computed without ever performing an NUFFT, which for datasets of 10^6
visibilities or more can be extremely computationally expensive.

This can make the likelihood function significantly faster, for example with speed ups of hundreds of times or more
for tens or millions of visibilities. In fact, the run time does not scale with the number of visibilities at all,
meaning datasets of any size can be fitted in seconds.

__Wrap Up__

We have presented a visual step-by-step guide to the pixelization likelihood function.

There are a number of other inputs features which slightly change the behaviour of this likelihood function, which
are described in additional notebooks found in this package. In brief, these describe:

 - **Over Sampling**: Oversampling the image grid into a finer grid of sub-pixels, which are all individually 
 paired fractionally with each rectangular pixel.

 - **Source-plane Interpolation**: Using bilinear interpolation on the rectangular pixelization to pair each 
 image (sub-)pixel to multiple rectangular pixels with interpolation weights.

 - **Luminosity Weighted Regularization**: Using an adaptive regularization coefficient which adapts the level of 
 regularization applied to the galaxy based on its luminosity.